In [1]:
# DCAT-3 Catalog and Dataset Generator for CIM/BIC

import os
import datetime
import pandas as pd
from sodapy import Socrata
from rdflib import Graph, Namespace, URIRef, Literal, BNode
from rdflib.namespace import RDF, DCTERMS, FOAF, XSD

# Define namespaces
dcat = Namespace("http://www.w3.org/ns/dcat#")
prov = Namespace("http://www.w3.org/ns/prov#")
spdx = Namespace("http://spdx.org/rdf/terms#")

# Initialize graph
g = Graph()
g.bind("dcat", dcat)
g.bind("dct", DCTERMS)
g.bind("foaf", FOAF)
g.bind("prov", prov)
g.bind("spdx", spdx)

# --- Configuration ---
cim_catalog_uri = URIRef("https://data.colorado.gov/catalog")
org_uri = URIRef("https://data.colorado.gov/organization/colorado-information-marketplace")
license_uri = URIRef("https://creativecommons.org/publicdomain/zero/1.0/")

# --- Organization ---
g.add((org_uri, RDF.type, FOAF.Organization))
g.add((org_uri, FOAF.name, Literal("Colorado Information Marketplace")))
g.add((org_uri, FOAF.homepage, URIRef("https://data.colorado.gov")))

# --- Catalog metadata ---
g.add((cim_catalog_uri, RDF.type, dcat.Catalog))
g.add((cim_catalog_uri, DCTERMS.title, Literal("Colorado Information Marketplace Catalog")))
g.add((cim_catalog_uri, DCTERMS.description, Literal("The official open data catalog for the State of Colorado.")))
g.add((cim_catalog_uri, DCTERMS.publisher, org_uri))
g.add((cim_catalog_uri, DCTERMS.issued, Literal("2024-07-01", datatype=XSD.date)))
g.add((cim_catalog_uri, DCTERMS.modified, Literal(datetime.date.today(), datatype=XSD.date)))
g.add((cim_catalog_uri, DCTERMS.license, license_uri))

# --- Retrieve example datasets from Socrata ---
with Socrata("data.colorado.gov", None) as client:
    datasets = client.datasets()

# Only use a few samples for demonstration
datasets = datasets[:5]

for ds in datasets:
    dsid = ds["resource"]["id"]
    dsuri = URIRef(f"https://data.colorado.gov/d/{dsid}")
    disturi = URIRef(f"https://data.colorado.gov/resource/{dsid}.csv")

    title = ds["resource"].get("name", "Untitled Dataset")
    desc = ds["resource"].get("description", "No description available.")
    mod = ds["resource"].get("metadata_updated_at", None)
    created = ds["resource"].get("createdAt", None)

    # --- Dataset ---
    g.add((dsuri, RDF.type, dcat.Dataset))
    g.add((dsuri, DCTERMS.title, Literal(title)))
    g.add((dsuri, DCTERMS.description, Literal(desc)))
    g.add((dsuri, DCTERMS.identifier, Literal(dsid)))
    g.add((dsuri, DCTERMS.publisher, org_uri))
    if created:
        g.add((dsuri, DCTERMS.issued, Literal(created.split('T')[0], datatype=XSD.date)))
    else:
        g.add((dsuri, DCTERMS.issued, Literal("2024-07-01", datatype=XSD.date)))

    if mod:
        g.add((dsuri, DCTERMS.modified, Literal(mod.split('T')[0], datatype=XSD.date)))
    else:
        g.add((dsuri, DCTERMS.modified, Literal(datetime.date.today(), datatype=XSD.date)))

    # --- Update frequency handling ---
    g.add((dsuri, DCTERMS.accrualPeriodicity, URIRef("http://purl.org/cld/freq/daily")))
    g.add((dsuri, dcat.temporalResolution, Literal("P1D", datatype=XSD.duration)))

    # --- Distribution ---
    g.add((disturi, RDF.type, dcat.Distribution))
    g.add((disturi, DCTERMS.title, Literal("CSV Download")))
    g.add((disturi, dcat.downloadURL, disturi))
    g.add((disturi, dcat.mediaType, Literal("text/csv")))
    g.add((disturi, DCTERMS.license, license_uri))

    g.add((dsuri, dcat.distribution, disturi))
    g.add((cim_catalog_uri, dcat.dataset, dsuri))

# --- Save TTL ---
outfile = "cim_dcat3.ttl"
g.serialize(destination=outfile, format="turtle")
print(f"✅ DCAT-3 file created: {outfile}")
g.serialize(destination="cim_dcat3.jsonld", format="json-ld", indent=2)


# --- Optional: Compliance check ---
missing_fields = []
for s, p, o in g.triples((None, RDF.type, dcat.Dataset)):
    for required in [DCTERMS.title, DCTERMS.description, dcat.distribution]:
        if not any(g.triples((s, required, None))):
            missing_fields.append((s, required))

if missing_fields:
    print("⚠️ Missing fields:")
    for s, f in missing_fields:
        print(f"- {s} missing {f}")
else:
    print("✅ All required DCAT fields present.")

✅ DCAT-3 file created: cim_dcat3.ttl
✅ All required DCAT fields present.


In [ ]:
from rdflib import Graph
from rdflib_neo4j import Neo4jStore, Neo4jStoreConfig

# 🧠 Your existing RDF graph 'g' already exists
# Example: g = Graph(); g.parse("cim_dcat3.ttl", format="turtle")

# 🔌 Connect to Neo4j
from rdflib_neo4j import Neo4jStore

uri = "bolt://10.255.255.254:7687"
user = "neo4j"
pwd = "444Jayla."
database = "neo4j"

# 🧱 Create the store
config = Neo4jStoreConfig(
    auth_data={
        "uri": "bolt://10.255.255.254:7687",
        "database": "neo4j",
        "user": "neo4j",
        "pwd": "444Jayla."
    }
)

# 🔌 Create the store using the config object
store = Neo4jStore(config)


# Create a Graph backed by Neo4j
# neo_graph = Graph(store=store)
# neo_graph.open("bolt://localhost:7687")

# 🚀 Push your RDF graph into Neo4j
neo_graph += g

# ✅ Always close the store
neo_graph.close()

print("✅ DCAT-3 RDF graph successfully loaded into Neo4j!")


ServiceUnavailable: Couldn't connect to 10.255.255.254:7687 (resolved to ('10.255.255.254:7687',)):
Failed to establish connection to ResolvedIPv4Address(('10.255.255.254', 7687)) (reason [Errno 111] Connection refused)